In [1]:
import os, sys, json, logging, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from scipy.signal import resample_poly
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score, f1_score

sys.path.append('../')
from sleeplib.config import Config
from sleeplib.datasets import BonoboDataset, Hardmine_BonoboDataset
from sleeplib.montages import CDAC_combine_montage
from sleeplib.transforms import cut_and_jitter, channel_flip, extremes_remover
from sleeplib.Resnet_15.model import ResNet


ModuleNotFoundError: No module named 'sleeplib'

In [ ]:
# load config and show all default parameters
config = Config()
config.print_config()
combine_montage = CDAC_combine_montage()

# load dataset
ROOT = '/egr/research-slim/luozishe/IEDClassification/SpikeNet2'
run_name = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
df = pd.read_csv(f'{ROOT}/hf_spikenet/lut_all.csv', sep=';')
checkpoints = f'{ROOT}/checkpoints/{run_name}'
run = f'{ROOT}/run/{run_name}'
os.makedirs(checkpoints, exist_ok=True)
os.makedirs(run, exist_ok=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path, run_path = checkpoints, run
MAX_EPOCHS, MIN_EPOCHS, PATIENCE = config.MAX_EPOCHS, config.MIN_EPOCHS, config.PATIENCE
BATCH = 32
log = logging.getLogger(run_name); log.setLevel(logging.INFO); log.handlers.clear()
for h in [logging.StreamHandler(sys.stdout), logging.FileHandler(f'{run_path}/log.txt')]:
    h.setFormatter(logging.Formatter('%(asctime)s %(message)s')); log.addHandler(h)

CLIPS    = config.PATH_FILES_BONOBO
PP_DIR   = f'{ROOT}/hf_spikenet/PerPatient'; os.makedirs(PP_DIR, exist_ok=True)
log.info(f'device {device} | run {run_name}')

In [ ]:
df['patient'] = df['eeg_file'].str.split('_Run').str[0]
NO_IED_DIR = Path('/egr/research-slim/shared/hf_dataset/CrossSubject_OUTPUT/no_ied')
ctrl_runs  = sorted(NO_IED_DIR.glob('*/Run*_meta.json'))
CTRL_PTS   = sorted({p.parent.name for p in ctrl_runs})
FP_PT, MINE_PTS = CTRL_PTS[-1], CTRL_PTS[:-1]
print(CTRL_PTS)
print('mine:', MINE_PTS, '| FP/hour:', FP_PT)
df.groupby('patient')['fraction_of_yes'].agg(n='size', n_ieds=lambda x: (x >= 0.5).sum())

In [ ]:
CLIP_SEC = 2.0
ORDER    = list(range(50))

def build_ctrl_lut(pts, stride_sec=0.25):
    rows = []
    for mpath in ctrl_runs:
        pt = mpath.parent.name
        if pt not in pts: continue
        meta = json.load(open(mpath))
        X = np.load(str(mpath).replace('_meta.json', '_X.npy'), mmap_mode='r')
        eeg_file = f"{pt}_Run{meta['run']}"
        fs, N = meta['sfreq'], meta['n_samples']
        half = int(CLIP_SEC / 2 * fs)
        for s in range(half, N - half, int(stride_sec * fs)):
            name = f'ctrl_{eeg_file}_{s}'
            f = f'{CLIPS}/{name}.npy'
            if not os.path.exists(f):
                seg = np.asarray(X[ORDER, s - half:s + half], np.float32)
                np.save(f, resample_poly(seg, 16, 25, axis=-1).astype(np.float32))
            rows.append((name, eeg_file, pt, s / fs))
    return pd.DataFrame(rows, columns=['event_file', 'eeg_file', 'patient', 't']).assign(
        total_votes_received=1, fraction_of_yes=0.0, Mode='Train')

ctrl_mine = build_ctrl_lut(MINE_PTS)      # 20529, 20535
ctrl_fp   = build_ctrl_lut([FP_PT])       # 20537
print(len(ctrl_mine), 'mining windows |', len(ctrl_fp), 'FP-eval windows')

lab = np.load(f"{CLIPS}/{df['event_file'].iloc[0]}.npy")
c   = np.load(f"{CLIPS}/{ctrl_mine['event_file'].iloc[0]}.npy")
print('labeled', lab.shape, '| control', c.shape)
print("ctrl_fp", ctrl_fp)
fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
ax[0].plot(lab[:5].T); ax[0].set_title('labeled clip'); ax[1].plot(c[:5].T); ax[1].set_title('control clip'); plt.show()

In [ ]:
print(len(df), 'labeled windows')
print(f"ctrl_mine: {len(ctrl_mine)} windows | ctrl_fp: {len(ctrl_fp)} windows")
print(f"ctrl_mine: {ctrl_mine['patient'].nunique()} patients | ctrl_fp: {ctrl_fp['patient'].nunique()} patients")

In [ ]:
PATIENTS = ['20551', '20556', '20558', '20559']

def split_fold(pid):
    d = pd.read_csv(f'{PP_DIR}/lut_{pid}.csv', sep=';')
    d['patient'] = d['eeg_file'].str.split('_Run').str[0]
    return [d[d['Mode'] == m] for m in ['Train', 'Val', 'Test']]

for pid in PATIENTS:
    tr, va, te = split_fold(pid)
    print(pid, *(f"{n}: {len(x)} clips / {(x['fraction_of_yes'] >= 0.5).sum()} IED"
                 for n, x in zip(['train', 'val', 'test'], [tr, va, te])), sep=' | ')

In [ ]:
def make_train_loader(d):
    d = d.sort_values('fraction_of_yes', ascending=False).reset_index(drop=True)
    ds = Hardmine_BonoboDataset(d, CLIPS, transform=transform_train_pos, transform_pos=transform_train_pos,
                                transform_neg=transform_train_neg, montage=combine_montage,
                                window_size=config.WINDOWSIZE, num_pos_augmentations=1)
    return DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=os.cpu_count())

def make_eval_loader(d):
    ds = BonoboDataset(d, CLIPS, transform=transform_val, montage=combine_montage, window_size=config.WINDOWSIZE)
    return DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=os.cpu_count())

In [ ]:
def run_train(loader, model, loss_fn, optimizer):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.float().to(device), y.float().to(device).view(-1)
        loss = loss_fn(model(x).view(-1), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)

def run_val(loader, model, loss_fn):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.float().to(device), y.float().to(device).view(-1)
            total += loss_fn(model(x).view(-1), y).item() * x.size(0)
    return total / len(loader.dataset)

def train_model(train_dataloader, val_dataloader, model, optimizer, loss_fn, tag='r0'):
    hist = []
    best_val = float('inf')
    bad = 0
    ckpt = f'{model_path}/best_{tag}.pt'               # one FILE, used for save, load and return
    model.to(device)
    for epoch in range(MAX_EPOCHS):
        tr = run_train(train_dataloader, model, loss_fn, optimizer)
        va = run_val(val_dataloader, model, loss_fn)
        hist.append((tr, va))
        save_curves(hist, run_path, tag)              # curves go to the run FOLDER (fine)
        log.info(f'[{tag}] epoch {epoch+1} | train {tr:.4f} | val {va:.4f}')
        if va < best_val or epoch == 0:               # epoch 0: always have a checkpoint
            best_val, bad = va, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if epoch + 1 >= MIN_EPOCHS and bad >= PATIENCE:
                log.info(f'[{tag}] early stop at epoch {epoch+1}'); break
    model.load_state_dict(torch.load(ckpt, map_location=device))   # back to the best epoch
    log.info(f'[{tag}] done | best val {best_val:.4f} | ckpt {ckpt}')
    return model, ckpt, hist

In [ ]:
transform_train_pos = transforms.Compose([cut_and_jitter(windowsize=config.WINDOWSIZE,max_offset=0.1,Fq=config.FQ), 
                                      channel_flip(p=0.5),
                                      extremes_remover(signal_max = 2000, signal_min = 20)
                                      #random_channel_zero(p=0.1),
                                      #noise_adder(s=0.1, p=0.1)
                                      ])

transform_train_neg = transforms.Compose([cut_and_jitter(windowsize=config.WINDOWSIZE,max_offset=0.1,Fq=config.FQ), 
                                      channel_flip(p=0.5),
                                      #random_channel_zero(p=0.1),
                                      #noise_adder(s=0.1, p=0.1),
                                      extremes_remover(signal_max = 2000, signal_min = 20)
                                      ])

transform_val = transforms.Compose([cut_and_jitter(windowsize=config.WINDOWSIZE,max_offset=0,Fq=config.FQ),
                                    extremes_remover(signal_max = 2000, signal_min = 20)])#,CDAC_signal_flip(p=0)])


In [ ]:
def cb_loss_fn(labels, beta, gamma=0.0):
    """Class-balanced (focal) BCE on probabilities: the model already applies a sigmoid."""
    n = np.bincount(np.asarray(labels), minlength=2)
    w = (1 - beta) / (1 - beta ** n)
    w = torch.tensor(2 * w / w.sum(), dtype=torch.float32, device=device)
    def loss(p, y):
        y = y.float(); p = p.clamp(1e-7, 1 - 1e-7)
        bce = F.binary_cross_entropy(p, y, reduction='none')
        pt = torch.where(y >= 0.5, p, 1 - p)
        return (w[(y >= 0.5).long()] * (1 - pt) ** gamma * bce).mean()
    return loss, w.cpu().numpy()

def save_curves(hist, path, tag):
    h = np.array(hist); plt.figure(figsize=(5, 3))
    plt.plot(h[:, 0], label='train'); plt.plot(h[:, 1], label='val'); plt.legend(); plt.title(tag)
    plt.savefig(f'{path}/loss_{tag}.png'); plt.close()

@torch.no_grad()
def predict(loader, model):
    model.eval(); P, Y = [], []
    for x, y in loader:
        P.append(model(x.float().to(device)).view(-1).cpu())
        Y.append(y.view(-1))
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def new_model():
    m = ResNet(lr=config.LR, n_channels=config.N_CHANNELS, Focal_loss=False).to(device)
    return m, torch.optim.Adam(m.parameters(), lr=config.LR)

m, _ = new_model(); x, _ = next(iter(make_eval_loader(df.head(64))))
out = m(x.float().to(device)).detach().cpu().view(-1)
print('output range:', out.min().item(), out.max().item())
plt.hist(out.numpy(), bins=30); plt.title('untrained ResNet outputs'); plt.show()

In [ ]:
def run_fold(k):
    tr, va, te = split_fold(k)
    val_dl, H, res, model = make_eval_loader(va), ctrl_mine.iloc[:0], [], None
    for r in range(len(TAUS) + 1):
        if r > 0:
            H = pd.concat([H, mine(model, set(H['event_file']), TAUS[r-1])], ignore_index=True)
            print(f'fold {k} round {r}: mined {len(H)} hard events')
        cur = pd.concat([tr, H], ignore_index=True)
        print(f'fold {k} round {r}: {len(cur)} train events')
        if USE_CB:
            loss_fn, w = cb_loss_fn((cur['fraction_of_yes'] >= 0.5).astype(int), beta=0.999)
        else:
            loss_fn, w = nn.BCELoss(), np.array([1.0, 1.0])          # plain BCE: equal weights
        log.info(f'fold {k} round {r}: {len(cur)} train | class weights {w.round(3)}')
        model, optimizer = new_model()
        model, ckpt, hist = train_model(make_train_loader(pd.concat([tr, H], ignore_index=True)),
                                        val_dl, model, optimizer, loss_fn, tag=f'{METHOD}_f{k}_r{r}')
        p, y = predict(val_dl, model)
        res.append(dict(fold=k, round=r, n_hard=len(H), val_auroc=roc_auc_score(y >= 0.5, p), ckpt=ckpt))
        log.info(f'fold {k} round {r}: |H|={len(H)} val AUROC {res[-1]["val_auroc"]:.3f}')
    return pd.DataFrame(res), va, te

In [ ]:
ctrl_loader, fp_loader = make_eval_loader(ctrl_mine), make_eval_loader(ctrl_fp)

def mine(model, H_names, rule):
    p, _ = predict(ctrl_loader, model)
    cand = ctrl_mine.assign(p=p)[~ctrl_mine['event_file'].isin(H_names)]
    new = cand[cand['p'] > rule] if isinstance(rule, float) else cand.nlargest(rule, 'p')   # float = τ (paper), int = top-K
    plt.hist(p, bins=50, log=True); plt.axvline(new['p'].min() if len(new) else rule, c='r')
    plt.title(f'rule={rule}: {len(new)} new'); plt.show()
    return new.drop(columns='p')

In [ ]:
def evaluate_fold(res, va, te, name=''):
    best = res.loc[res['val_auroc'].idxmax()]
    model, _ = new_model(); model.load_state_dict(torch.load(best['ckpt'], map_location=device))
    pv, yv = predict(make_eval_loader(va), model); fpr_v, tpr_v, th = roc_curve(yv >= 0.5, pv)
    t = th[np.argmax(tpr_v - fpr_v)]
    pt, yt = predict(make_eval_loader(te), model); yt = yt >= 0.5; hat = pt >= t
    pf, _ = predict(fp_loader, model)
    k = best['fold']
    out = dict(fold=k, best_round=int(best['round']), thr=t,
               auroc=roc_auc_score(yt, pt), auprc=average_precision_score(yt, pt),
               sens=(hat & yt).sum() / yt.sum(), spec=(~hat & ~yt).sum() / (~yt).sum(),
               f1=f1_score(yt, hat), fp_per_hr=(pf >= t).sum() / (len(ctrl_fp) * 0.25 / 3600))

    label = f"{name}, round {out['best_round']}" if name else f"round {out['best_round']}"
    fpr, tpr, _ = roc_curve(yt, pt); prec, rec, _ = precision_recall_curve(yt, pt)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(fpr, tpr, label=f"AUROC {out['auroc']:.3f}"); ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax[0].scatter(1 - out['spec'], out['sens'], c='r', zorder=3, label=f'threshold {t:.2f}')
    ax[0].set_xlabel('false positive rate'); ax[0].set_ylabel('true positive rate')
    ax[1].plot(rec, prec, label=f"AUPRC {out['auprc']:.3f}")
    ax[1].axhline(yt.mean(), c='k', ls='--', lw=0.8, label=f'chance {yt.mean():.2f}')
    ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision')
    for a in ax: a.legend(loc='lower right' if a is ax[0] else 'upper right'); a.set_xlim(0, 1); a.set_ylim(0, 1.02)
    fig.suptitle(f'{METHOD} | fold {k} test | {label}', fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{run_path}/{METHOD}_test_roc_f{k}_{name.replace(' ', '_') or 'eval'}.png"); plt.show()

    out['_curve'] = (fpr, tpr)
    return out

In [ ]:
USE_CB = False
METHOD = 'pp_sanity'                 # keep sanity runs separate from the real ones
TAUS   = [20]

pid = '20551'
res0, va0, te0 = run_fold(pid)
display(res0)
display(evaluate_fold(res0, va0, te0))
plt.imshow(plt.imread(f'{run_path}/loss_{METHOD}_f{pid}_r0.png')); plt.axis('off'); plt.show()

In [ ]:
USE_CB, METHOD, TAUS = True, 'pp_thr_cb', [0.8, 0.7, 0.6, 0.5]
# other runs:
# USE_CB, METHOD, TAUS = True,  'pp_topk_cb',  [20, 40, 80]
# USE_CB, METHOD, TAUS = False, 'pp_thr_bce',  [0.8, 0.7, 0.6, 0.5]
# USE_CB, METHOD, TAUS = False, 'pp_topk_bce', [20, 40, 80]

all_res, metrics, metrics_r0 = [], [], []
for k in PATIENTS:
    res, va, te = run_fold(k)
    all_res.append(res)
    metrics.append(evaluate_fold(res, va, te, 'with mining'))
    metrics_r0.append(evaluate_fold(res.iloc[[0]], va, te, 'no mining'))
    pd.concat(all_res).to_csv(f'{run_path}/{METHOD}_rounds.csv', index=False)
    for name, m in [(METHOD, metrics), (f'{METHOD}_nomine', metrics_r0)]:
        pd.DataFrame(m).drop(columns='_curve').to_csv(f'{run_path}/{name}_cv5_metrics.csv', index=False)

# summary: no mining vs mining
cols = ['auroc', 'auprc', 'sens', 'spec', 'f1', 'fp_per_hr']
S = pd.DataFrame({name: pd.DataFrame(m)[cols].agg(['mean', 'std']).round(3).apply(lambda c: f"{c['mean']} ± {c['std']}")
                  for name, m in [('no mining', metrics_r0), (METHOD, metrics)]}).T
display(S)
print('best round per fold:', [m['best_round'] for m in metrics])

grid = np.linspace(0, 1, 201)

def mean_roc(ms):
    tprs = np.array([np.interp(grid, *m['_curve']) for m in ms])
    return tprs.mean(0), tprs.std(0), np.mean([m['auroc'] for m in ms])

plt.figure(figsize=(6.5, 5))
for m in metrics:                                           # per-fold curves (with mining), as before
    plt.plot(*m['_curve'], lw=0.8, alpha=0.5, label=f"fold {m['fold']} ({m['auroc']:.2f})")
for ms, name, c in [(metrics, f'mean, {METHOD} mining', 'k'), (metrics_r0, 'mean, no mining', 'tab:red')]:
    mean, sd, auc_m = mean_roc(ms)
    plt.plot(grid, mean, c=c, lw=2, label=f'{name} {auc_m:.3f}')
    plt.fill_between(grid, mean - sd, mean + sd, color=c, alpha=0.1)
plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
plt.xlabel('false positive rate'); plt.ylabel('true positive rate'); plt.title(f'Per-patient test ROC ({METHOD} vs no mining)')
plt.legend(fontsize=8); plt.savefig(f'{run_path}/{METHOD}_test_roc_cv5_vs_nomine.png'); plt.show()